In [35]:
import pandas as pd
import numpy as np
import random
import featuretools as ft
import woodwork
from dotenv import load_dotenv
import os
import mlflow
import joblib
import json
from mlflow.tracking import MlflowClient

In [2]:

with open('fastapi/replacer.json', 'r', encoding='utf-8') as f:
    replacer_loaded = json.load(f)


In [3]:
replacer_loaded

{'ind_empleado': {'A': 'other', 'B': 'other', 'F': 'other', 'S': 'other'},
 'pais_residencia': {'CA': 'other',
  'CH': 'other',
  'CL': 'other',
  'IE': 'other',
  'AT': 'other',
  'NL': 'other',
  'FR': 'other',
  'GB': 'other',
  'DE': 'other',
  'DO': 'other',
  'BE': 'other',
  'AR': 'other',
  'VE': 'other',
  'US': 'other',
  'MX': 'other',
  'BR': 'other',
  'IT': 'other',
  'EC': 'other',
  'PE': 'other',
  'CO': 'other',
  'HN': 'other',
  'FI': 'other',
  'SE': 'other',
  'AL': 'other',
  'PT': 'other',
  'MZ': 'other',
  'CN': 'other',
  'TW': 'other',
  'PL': 'other',
  'IN': 'other',
  'CR': 'other',
  'NI': 'other',
  'HK': 'other',
  'AD': 'other',
  'CZ': 'other',
  'AE': 'other',
  'MA': 'other',
  'GR': 'other',
  'PR': 'other',
  'RO': 'other',
  'IL': 'other',
  'RU': 'other',
  'GT': 'other',
  'GA': 'other',
  'NO': 'other',
  'SN': 'other',
  'MR': 'other',
  'UA': 'other',
  'BG': 'other',
  'PY': 'other',
  'EE': 'other',
  'SV': 'other',
  'ET': 'other',
  'CM

In [10]:
df = pd.read_csv(r'data/train_ver2.csv')
try:
    df = df.drop(columns=['Unnamed: 0'])
except:
    pass
df

C:\Users\Admin\AppData\Local\Temp\ipykernel_4904\266042898.py:1: DtypeWarning: Columns (6,9,12,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'data/train_df.csv')


,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,indrel,...,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
0,2015-01-28,1375586,N,ES,H,35,2015-01-12,0.0,6,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
1,2015-01-28,1050611,N,ES,V,23,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
2,2015-01-28,1050612,N,ES,V,23,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
3,2015-01-28,1050613,N,ES,H,22,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
4,2015-01-28,1050614,N,ES,V,23,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13647304,2016-05-28,1166765,N,ES,V,22,2013-08-14,0.0,33,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
13647305,2016-05-28,1166764,N,ES,V,23,2013-08-14,0.0,33,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
13647306,2016-05-28,1166763,N,ES,H,47,2013-08-14,0.0,33,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
13647307,2016-05-28,1166789,N,ES,H,22,2013-08-14,0.0,33,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0


# Сгенерируем случайный профайл клиента

In [11]:
random_profile = []
for c in df.columns:
    random_element = random.choice(df[c].unique())
    random_profile.append(random_element)
rf = pd.DataFrame(columns=df.columns)
rf.loc[0] = random_profile
rf

,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,indrel,...,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
0,2015-03-28,1099600,N,AE,V,86,2012-04-06,NaN,25,1.0,...,0,1,0,1,0,1,1,NaN,NaN,1


In [12]:
di =dict(zip(rf.columns,random_profile))
di

{'fecha_dato': '2015-03-28',
 'ncodpers': np.int64(1099600),
 'ind_empleado': 'N',
 'pais_residencia': 'AE',
 'sexo': 'V',
 'age': 86,
 'fecha_alta': '2012-04-06',
 'ind_nuevo': np.float64(nan),
 'antiguedad': '     25',
 'indrel': np.float64(1.0),
 'ult_fec_cli_1t': '2015-11-12',
 'indrel_1mes': 4.0,
 'tiprel_1mes': nan,
 'indresi': 'N',
 'indext': nan,
 'conyuemp': 'N',
 'canal_entrada': 'KHA',
 'indfall': 'S',
 'tipodom': np.float64(1.0),
 'cod_prov': np.float64(33.0),
 'nomprov': 'ALAVA',
 'ind_actividad_cliente': np.float64(1.0),
 'renta': np.float64(132221.88),
 'segmento': '01 - TOP',
 'ind_ahor_fin_ult1': np.int64(1),
 'ind_aval_fin_ult1': np.int64(0),
 'ind_cco_fin_ult1': np.int64(0),
 'ind_cder_fin_ult1': np.int64(1),
 'ind_cno_fin_ult1': np.int64(1),
 'ind_ctju_fin_ult1': np.int64(1),
 'ind_ctma_fin_ult1': np.int64(0),
 'ind_ctop_fin_ult1': np.int64(1),
 'ind_ctpp_fin_ult1': np.int64(1),
 'ind_deco_fin_ult1': np.int64(0),
 'ind_deme_fin_ult1': np.int64(0),
 'ind_dela_fin_ult

# Совершим предобработку

In [13]:
cats = [x for x in list(df.columns) if x not in ['age','antiguedad','fecha_dato','ncodpers','renta']]
di = {}
for col in cats:
    di.update({col:df[col].mode()[0]})
di

{'ind_empleado': 'N',
 'pais_residencia': 'ES',
 'sexo': 'V',
 'fecha_alta': '2014-07-28',
 'ind_nuevo': np.float64(0.0),
 'indrel': np.float64(1.0),
 'ult_fec_cli_1t': '2015-12-24',
 'indrel_1mes': 1.0,
 'tiprel_1mes': 'I',
 'indresi': 'S',
 'indext': 'N',
 'conyuemp': 'N',
 'canal_entrada': 'KHE',
 'indfall': 'N',
 'tipodom': np.float64(1.0),
 'cod_prov': np.float64(28.0),
 'nomprov': 'MADRID',
 'ind_actividad_cliente': np.float64(0.0),
 'segmento': '02 - PARTICULARES',
 'ind_ahor_fin_ult1': np.int64(0),
 'ind_aval_fin_ult1': np.int64(0),
 'ind_cco_fin_ult1': np.int64(1),
 'ind_cder_fin_ult1': np.int64(0),
 'ind_cno_fin_ult1': np.int64(0),
 'ind_ctju_fin_ult1': np.int64(0),
 'ind_ctma_fin_ult1': np.int64(0),
 'ind_ctop_fin_ult1': np.int64(0),
 'ind_ctpp_fin_ult1': np.int64(0),
 'ind_deco_fin_ult1': np.int64(0),
 'ind_deme_fin_ult1': np.int64(0),
 'ind_dela_fin_ult1': np.int64(0),
 'ind_ecue_fin_ult1': np.int64(0),
 'ind_fond_fin_ult1': np.int64(0),
 'ind_hip_fin_ult1': np.int64(0),
 

In [14]:
rf = rf.drop(['indrel','indext'],axis=1) 
for col in ['age','antiguedad','renta']:
    rf[col] = pd.to_numeric(rf[col], errors='coerce')
    df[col] = pd.to_numeric(df[col], errors='coerce')
rf['antiguedad']=rf['antiguedad'].replace(np.float64(-999999.0),np.nan)
# заменяем пропуски в числовых признаках на медиану
for nc in ['age', 'antiguedad','renta']:
    rf.fillna({nc: df[nc].median()}, inplace=True)
# заменяем пропуски в категориальных признаках на самое частое значение
cats = [x for x in list(df.columns) if x not in ['age','antiguedad','fecha_dato','ncodpers','renta']]
for col in cats:
    rf.fillna({col: df[col].mode()[0]}, inplace=True)
intervals = [(2, 24), (25, 29), (30, 37), (38, 42), (43, 46), (47, 50), (51, 55), (56, 64), (65, 164)]
rf['age_interval'] = pd.cut(rf['age'], bins=[interval[0] for interval in intervals] + [intervals[-1][1]], 
                              labels=[f"{interval[0]}-{interval[1]}" for interval in intervals])
rf


,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,ult_fec_cli_1t,...,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1,age_interval
0,2015-03-28,1099600,N,AE,V,86,2012-04-06,0.0,25,2015-11-12,...,1,0,1,0,1,1,0.0,0.0,1,65-164


In [15]:
# Define the bin edges (ensure these match your intervals exactly)
bin_edges1 = [
    '1995-01-15', 
    '2000-05-19', 
    '2002-02-19', 
    '2004-04-28', 
    '2006-07-18', 
    '2008-10-04',   
    '2011-09-06', 
    '2012-07-23', 
    '2012-12-14',
    '2013-10-21', 
    '2014-08-11', 
    '2015-02-26', 
    '2016-05-31'
]
bin_edges2 = [
    '2015-06-30',
    '2015-12-24',
    '2016-05-30'
    #[(2015-06-30 00:00:00, 2015-12-24 00:00:00] < (2015-12-24 00:00:00, 2016-05-30 00:00:00]]
]

In [16]:
bin_edges1 = pd.to_datetime(bin_edges1)
bin_edges2 = pd.to_datetime(bin_edges2)

In [17]:
rf['fecha_alta'] = pd.cut(
        rf['fecha_alta'], 
        bins=bin_edges1,
        right=True,       # Intervals are closed on the right (e.g., 2000-05-19 is included in the first bin)
        ordered=True      # Maintain logical order of intervals
    )
rf['ult_fec_cli_1t'] = pd.cut(
        rf['ult_fec_cli_1t'], 
        bins=bin_edges2,
        right=True,       # Intervals are closed on the right (e.g., 2000-05-19 is included in the first bin)
        ordered=True      # Maintain logical order of intervals
    )

In [18]:
cats = [x for x in list(rf.columns) if x not in ['age','antiguedad','fecha_dato','ncodpers','renta']]
threshold = 0.01
for col in cats[:-24]:
    # Считаем количество вхождений каждого значения в столбце 
    value_counts = df[col].value_counts(normalize=True)
    #print(value_counts)
    # Определяем редкие значения, которые встречаются реже, чем threshold
    rare_values = value_counts[value_counts < threshold].index
    if rf[col].isin(rare_values).sum() > 0:
        rf[col]='other'

In [19]:
rf['renta'] = df['renta'].clip(lower=26449.65, upper=337117.17)
rf['antiguedad'] = df['antiguedad'].clip(lower=1.0, upper=207.0)

In [24]:
personal_recs = pd.read_parquet('fastapi/personal_als.parquet')
try:
    recommended_product_id = personal_recs.loc[rd['ncodpers'],'recommended_product_id']
except:
    recommended_product_id = None
rf['recommended_product_id'] = recommended_product_id
rf['recommended_product_id']= rf['recommended_product_id'].fillna('0')

In [25]:
products=['ind_ahor_fin_ult1',
'ind_aval_fin_ult1',
'ind_cco_fin_ult1',
'ind_cder_fin_ult1',
'ind_cno_fin_ult1',
'ind_ctju_fin_ult1',
'ind_ctma_fin_ult1',
'ind_ctop_fin_ult1',
'ind_ctpp_fin_ult1',
'ind_deco_fin_ult1',
'ind_deme_fin_ult1',
'ind_dela_fin_ult1',
'ind_ecue_fin_ult1',
'ind_fond_fin_ult1',
'ind_hip_fin_ult1',
'ind_plan_fin_ult1',
'ind_pres_fin_ult1',
'ind_reca_fin_ult1',
'ind_tjcr_fin_ult1',
'ind_valo_fin_ult1',
'ind_viv_fin_ult1',
'ind_nomina_ult1',
'ind_nom_pens_ult1',
'ind_recibo_ult1']


In [26]:
for c in products:
    rf[c] = pd.to_numeric(rf[c], errors='coerce')

In [27]:
rf['total_products'] = rf[products].sum(axis=1)

In [28]:
rf =rf[['ind_empleado', 'pais_residencia', 'sexo', 'ind_nuevo', 'antiguedad',
       'indrel_1mes', 'tiprel_1mes', 'indresi', 'conyuemp', 'canal_entrada',
       'indfall', 'cod_prov', 'ind_actividad_cliente', 'renta', 'segmento',
       'ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1',
       'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1',
       'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1',
       'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1',
       'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1',
       'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1',
       'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1',
       'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1',
       'total_products', 'age_interval', 'recommended_product_id']]

In [29]:
def feature_engineering(df):
    """Автоматическая генерация признаков для всех типов переменных"""
    
    # Сохраняем оригинальные данные
    original_df = df.copy()
    
    # Создаем EntitySet
    es = ft.EntitySet(id='bank_data')
    es = es.add_dataframe(
        dataframe_name='main',
        dataframe=df,                # Исходный DataFrame без reset_index()
        index='unique_id',           # Название для нового индекса
        make_index=True,             # Сгенерировать уникальный индекс
        logical_types={
            'antiguedad': woodwork.logical_types.Double,
            'renta': woodwork.logical_types.Double,
            **{col: woodwork.logical_types.Categorical for col in df.columns 
            if col not in ['antiguedad', 'renta']}  # Убрали 'index' из исключений
        }
    )


    # Определяем примитивы для разных типов признаков
    trans_primitives = [
        'add_numeric',
        'multiply_numeric',
        'divide_numeric',
        'natural_logarithm',  # Правильное название для log
        'square_root'         # Правильное название для sqrt
    ]
    
    agg_primitives = [
        'mean',
        'median',
        'std',
        'max',
        'min',
        'count',
        'num_unique'
    ]

    # Генерация признаков
    feature_matrix, _ = ft.dfs(
        entityset=es,
        target_dataframe_name='main',
        trans_primitives=trans_primitives,
        agg_primitives=agg_primitives,
        where_primitives=['count'],
        max_depth=2,
        features_only=False,
        verbose=True
    )

    # Объединение признаков
    try:
        new_features = feature_matrix.drop(columns=['unique_id'])
    except:
        new_features = feature_matrix
    try:
        new_features = feature_matrix.drop(columns=['index'])
    except:
        new_features = feature_matrix
    #df = pd.concat([original_df, new_features], axis=1)
    df = new_features
    #print(f'Новые признаки: {new_features.columns}')
    # Ручные трансформации
    df = manual_transformations(df)
    
    # Удаление дубликатов
    df = df.loc[:, ~df.columns.duplicated()]
    
    return df

def manual_transformations(df):
    """Ручные преобразования и кастомные фичи"""
    # Для числовых
    #print(df.columns)
    df['renta_antiguedad_ratio'] = df['renta'] / (df['antiguedad'] + 1)
    df['log_renta'] = np.log1p(df['renta'])
    
    # Для категориальных
    for col in ['pais_residencia', 'segmento']:
        df[f'mean_renta_by_{col}'] = df.groupby(col)['renta'].transform('mean')
        df[f'median_antiguedad_by_{col}'] = df.groupby(col)['antiguedad'].transform('median')
    
    # Взаимодействие категорий
    df['renta_vs_country_mean'] = df['renta'] / df['mean_renta_by_pais_residencia']
    
    return df

In [30]:
rf['antiguedad'] = rf['antiguedad'].astype(int)
rf = feature_engineering(rf)

c:\Users\Admin\miniconda3\Lib\site-packages\featuretools\synthesis\deep_feature_synthesis.py:169: UserWarning: Only one dataframe in entityset, changing max_depth to 1 since deeper features cannot be created
  warnings.warn(
c:\Users\Admin\miniconda3\Lib\site-packages\featuretools\synthesis\dfs.py:321: UnusedPrimitiveWarning: Some specified primitives were not used during DFS:
  agg_primitives: ['count', 'max', 'mean', 'median', 'min', 'num_unique', 'std']
  where_primitives: ['count']
This may be caused by a using a value of max_depth that is too small, not setting interesting values, or it may indicate no compatible columns for the primitive were found in the data. If the DFS call contained multiple instances of a primitive in the list above, none of them were used.
  warnings.warn(warning_msg, UnusedPrimitiveWarning)


Built 50 features
Elapsed: 00:00 | Progress: 100%|██████████


C:\Users\Admin\AppData\Local\Temp\ipykernel_4904\1894221252.py:83: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f'mean_renta_by_{col}'] = df.groupby(col)['renta'].transform('mean')
C:\Users\Admin\AppData\Local\Temp\ipykernel_4904\1894221252.py:84: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f'median_antiguedad_by_{col}'] = df.groupby(col)['antiguedad'].transform('median')
C:\Users\Admin\AppData\Local\Temp\ipykernel_4904\1894221252.py:83: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior 

In [36]:
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = "Bank_product_Modeling"

# Локальный tracking-сервер MLflow
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
client = MlflowClient()



In [37]:
# 1. Получаем эксперимент по имени
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f"Эксперимент '{EXPERIMENT_NAME}' не найден")

# 2. Ищем все run'ы с логированной моделью, сортируем по времени старта (свежие — первыми)
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["attributes.start_time DESC"],
    max_results=100,
)

model = None
for run in runs:
    artifacts = client.list_artifacts(run.info.run_id)
    # Ищем артефакт типа "model" (стандартный путь mlflow.log_model / log_model sklearn и т.п.)
    model_artifacts = [a for a in artifacts if a.path == "model"]
    if model_artifacts:
        model_uri = f"runs:/{run.info.run_id}/model"
        model = mlflow.pyfunc.load_model(model_uri)
        print(f"Загружена модель из run_id={run.info.run_id}, "
              f"start_time={run.info.start_time}")
        break

if model is None:
    raise ValueError("В эксперименте не найдено ни одной сохранённой модели")


Загружена модель из run_id=2b4133e392cc4a32a49cd608d3c46336, start_time=1789472161659


In [38]:
nums = ['antiguedad', 'renta', 'antiguedad + renta', 'antiguedad / renta', 'renta / antiguedad', 'antiguedad * renta', 'NATURAL_LOGARITHM(antiguedad)', 'NATURAL_LOGARITHM(renta)', 'SQUARE_ROOT(antiguedad)', 'SQUARE_ROOT(renta)', 'renta_antiguedad_ratio', 'log_renta', 'renta_vs_country_mean']
cats = ['ind_empleado', 'pais_residencia', 'sexo', 'ind_nuevo', 'indrel_1mes', 'tiprel_1mes', 'indresi', 'conyuemp', 'canal_entrada', 'indfall', 'cod_prov', 'ind_actividad_cliente', 'segmento', 'ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1', 'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1', 'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1', 'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1', 'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1', 'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1', 'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1', 'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1', 'total_products', 'age_interval', 'recommended_product_id', 'mean_renta_by_pais_residencia', 'median_antiguedad_by_pais_residencia', 'mean_renta_by_segmento', 'median_antiguedad_by_segmento']
for nu in nums:
    rf[nu]=pd.to_numeric(rf[nu], errors='coerce')
for cat in cats:
    rf[cat]=rf[cat].astype('str')

In [39]:
pd.set_option('display.max_columns', None)
rf

,ind_empleado,pais_residencia,sexo,ind_nuevo,antiguedad,indrel_1mes,tiprel_1mes,indresi,conyuemp,canal_entrada,indfall,cod_prov,ind_actividad_cliente,renta,segmento,ind_ahor_fin_ult1,ind_aval_fin_ult1,ind_cco_fin_ult1,ind_cder_fin_ult1,ind_cno_fin_ult1,ind_ctju_fin_ult1,ind_ctma_fin_ult1,ind_ctop_fin_ult1,ind_ctpp_fin_ult1,ind_deco_fin_ult1,ind_deme_fin_ult1,ind_dela_fin_ult1,ind_ecue_fin_ult1,ind_fond_fin_ult1,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1,total_products,age_interval,recommended_product_id,antiguedad + renta,antiguedad / renta,renta / antiguedad,antiguedad * renta,NATURAL_LOGARITHM(antiguedad),NATURAL_LOGARITHM(renta),SQUARE_ROOT(antiguedad),SQUARE_ROOT(renta),renta_antiguedad_ratio,log_renta,mean_renta_by_pais_residencia,median_antiguedad_by_pais_residencia,mean_renta_by_segmento,median_antiguedad_by_segmento,renta_vs_country_mean
unique_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,N,other,V,0.0,6.0,other,I,other,N,other,other,33.0,1.0,87218.1,01 - TOP,nan,0,0,1,1,1,0,1,1,0,0,0,1,0,0,1,0,1,0,1,1,0.0,0.0,1,11.0,65-164,0,87224.1,0.000069,14536.35,523308.6,1.791759,11.376167,2.44949,295.327107,12459.728571,11.376179,87218.1,6.0,87218.1,6.0,1.0


In [42]:
pr =int(model.predict(rf))
print(f'Рекомендован товар: {pr}:{products[pr]}')

Рекомендован товар: 0:ind_ahor_fin_ult1


C:\Users\Admin\AppData\Local\Temp\ipykernel_4904\1932366765.py:1: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pr =int(model.predict(rf))
